# Token Representation Tracking

Track how individual token representations evolve through the model:
identity trajectory, divergence, mixing rate, velocity, and convergence.

## Why This Matters

Token representation tracking tracks how individual token representations evolve through the network. Understanding token-level dynamics reveals how context is integrated, when predictions form, and how different positions interact to produce the output.

**Key references:**
- [Elhage et al. (2021) "A Mathematical Framework for Transformer Circuits"](https://transformer-circuits.pub/2021/framework/index.html) — Foundational framework for attention head and residual stream analysis

In [ ]:
import jax
import jax.numpy as jnp
from irtk import HookedTransformer, HookedTransformerConfig
from irtk.token_representation_tracking import (
    token_identity_trajectory, position_representation_divergence,
    token_mixing_rate, representation_velocity,
    representation_convergence,
)

cfg = HookedTransformerConfig(
    n_layers=4, d_model=32, n_ctx=64, d_head=8, n_heads=4, d_vocab=100,
)
model = HookedTransformer(cfg)
key = jax.random.PRNGKey(42)
leaves, treedef = jax.tree.flatten(model)
new_leaves = []
for leaf in leaves:
    if isinstance(leaf, jnp.ndarray) and leaf.dtype == jnp.float32:
        key, subkey = jax.random.split(key)
        new_leaves.append(jax.random.normal(subkey, leaf.shape) * 0.1)
    else:
        new_leaves.append(leaf)
model = jax.tree.unflatten(treedef, new_leaves)
tokens = jnp.array([1, 15, 30, 45, 60, 75, 90])
print('Model ready')

## Token Identity Trajectory

How does a token's representation change through the model?

In [ ]:
result = token_identity_trajectory(model, tokens, position=-1)
print(f"Position {result['position']}, original token: {result['original_token']}")
print(f"Final prediction: {result['final_prediction']}")
print(f"Identity decay: {result['identity_decay']:.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: norm={p['norm']:.4f}, "
          f"top={p['top_token']}, conf={p['confidence']:.4f}, "
          f"embed_sim={p['embed_similarity']:+.4f}")

## Position Representation Divergence

Do positions become more or less similar through layers?

In [ ]:
result = position_representation_divergence(model, tokens)
div = 'DIVERGE' if result['representations_diverge'] else 'converge'
print(f"Trend: {result['divergence_trend']:+.4f} ({div})\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: cos_sim={p['mean_cosine_similarity']:.4f}, "
          f"l2_dist={p['mean_l2_distance']:.4f}, divergence={p['divergence']:.4f}")

## Token Mixing Rate

How fast does a token's representation mix with the mean?

In [ ]:
result = token_mixing_rate(model, tokens, position=-1)
print(f"Position {result['position']}, mixing rate: {result['mixing_rate']:+.4f}\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: align_mean={p['alignment_to_mean']:+.4f}, "
          f"uniqueness={p['uniqueness']:.4f}, norm={p['norm']:.4f}")

## Representation Velocity

How fast do representations change between layers?

In [ ]:
result = representation_velocity(model, tokens)
print(f"Mean velocity: {result['mean_velocity']:.4f}")
print(f"Peak layer: {result['peak_layer']}\n")
for p in result['per_layer']:
    bar = '█' * int(p['mean_velocity'] * 20) if p['mean_velocity'] > 0 else ''
    print(f"  Layer {p['layer']}: vel={p['mean_velocity']:.4f}, "
          f"max={p['max_velocity']:.4f} (pos {p['fastest_position']}) {bar}")

## Representation Convergence

Do all token representations converge to similar values?

In [ ]:
result = representation_convergence(model, tokens)
conv = 'CONVERGE' if result['representations_converge'] else 'diverge'
print(f"Early similarity: {result['early_similarity']:.4f}")
print(f"Late similarity: {result['late_similarity']:.4f}")
print(f"Convergence: {result['convergence']:+.4f} [{conv}]\n")
for p in result['per_layer']:
    print(f"  Layer {p['layer']}: pairwise_sim={p['mean_pairwise_similarity']:.4f}")